# Retail Shopping Assistant

## Notebook 5: Performance Measurement on H100

Replay Notebook 4's trace against Nemotron 3.5 Super on vLLM with
[AIPerf](https://github.com/ai-dynamo/aiperf). Measure, at each concurrency:

- **Prefix-cache hit rate:** the share of prompt tokens the server did not
  recompute. From vLLM's own counters.
- **Time to first token (TTFT):** p50 and p95.
- **Throughput:** requests and output tokens per second.

Then the same with caching off: what caching is worth.

**Runs on the GPU machine alone.** No agent, catalog, memory, or Phoenix.

### Before you start

All on the GPU machine, from the repo root. Details and other machines:
[`gpu/README.md`](gpu/README.md).

1. **Serve the model** (4x H100 by default; first run downloads ~240 GB):

```bash
export HF_TOKEN=hf_...
notebook/gpu/deploy.sh
```

2. **Prove the cache works:** `notebook/gpu/verify_prefix_cache.sh`. Want a
   speedup above ~2x. On this hybrid model the hit rate alone can look healthy
   while nothing is saved.
3. **AIPerf and Jupyter**, once. With [uv](https://docs.astral.sh/uv/), which
   needs no `sudo`; stock Ubuntu images lack `python3-venv`, so
   `python3 -m venv` fails there:

```bash
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv venv ~/aiperf --python 3.12
uv pip install --python ~/aiperf/bin/python aiperf==0.13.0 jupyterlab matplotlib
~/aiperf/bin/jupyter lab --no-browser --port 8888 notebook/
```

   Open this notebook in *that* Jupyter. Another Jupyter on the machine
   (a cloud image's own) lacks `matplotlib`; Step 3 finds AIPerf at
   `~/aiperf/bin/aiperf` either way.

4. **From a laptop:** `ssh -L 8888:localhost:8888 <gpu-host>`, then open the
   link Jupyter printed. Keep AIPerf on the GPU machine: through an SSH tunnel,
   the tunnel becomes the bottleneck.

- **The trace:** `notebook/traces/`, committed by Notebook 4. Nothing to copy.
- **Rehearse without a GPU:** `pip install "git+https://github.com/ai-dynamo/aiperf.git@v0.11.0#subdirectory=tests/aiperf_mock_server"`,
  run `aiperf-mock-server --port 8765`, and set
  `BASE_URL=http://localhost:8765 METRICS_URL=http://localhost:8765/vllm/metrics`.

## Step 1: Check the Server

- Ready, which model, and how vLLM resolved caching.

In [ ]:
import os

from stress_replay import api_key, server_status

BASE_URL = os.environ.get("BASE_URL", "http://localhost:8000")
METRICS_URL = os.environ.get("METRICS_URL", f"{BASE_URL}/metrics")
KEY = api_key()

server = server_status(BASE_URL, METRICS_URL, KEY)
MODEL = server["models"][0] if server["models"] else None
CACHE = {"True": "on", "False": "off"}.get(server["enable_prefix_caching"], "unknown")
BLOCK = server["block_size"] or "na"
print(f"ready: {server['ready']}   model: {MODEL}   api key: {'set' if KEY else 'none'}")
print(f"prefix caching: {CACHE}   block_size: {BLOCK}   mamba_cache_mode: {server['mamba_cache_mode']}")

**Expect:**

```
ready: True   model: nemotron-3.5-super   api key: set
prefix caching: on   block_size: 2112   mamba_cache_mode: align
```

- `block_size: 2112`, not the 64 `deploy.sh` asked for: normal on this model.
  vLLM grows the attention block until it is as large as one Mamba state
  (`docker logs nemotron35` says so). Prompts are still matched every 64
  tokens. The exact number depends on the model and `TP`.
- `ready: False`: still loading (`docker logs -f nemotron35`), or not on `BASE_URL`.
- `prefix caching: off` when you want it on: rerun `deploy.sh` without
  `PREFIX_CACHE=0`.
- `mamba_cache_mode` not `align`: only attention is cached. Hit rates will look
  fine; TTFT will not improve.
- `unknown`: the mock server, or not vLLM. The hit rate stays empty.

### The GPUs the Model Runs On

- Live bars for the GPUs holding the model (more than 1 GiB in use) for 20
  seconds, then their averages. Idle GPUs on the machine are left out.
- **Busy** is the share of time any kernel ran: near 100% even for small
  batches. **Power** against the limit says how hard they work.
- Idle, expect little busy time and power well under the limit. Step 4 shows
  the same bars during every run.

In [ ]:
from stress_replay import gpu_dashboard

gpus = gpu_dashboard(seconds=20)

## Step 2: Load the Trace

- Reads `notebook/traces/`: the requests, and what was recorded for each.
- Compares the capture's model with the server's.

In [ ]:
import json

from stress_replay import TRACES, read_jsonl

capture = json.loads((TRACES / "capture.json").read_text())
trace = read_jsonl(TRACES / "journeys.jsonl.xz")
manifest = read_jsonl(TRACES / "manifest.jsonl.xz")
journeys = sorted({line["session_id"] for line in trace})

print(f"trace: {len(trace)} calls, {len(journeys)} journeys, captured {capture['since'][:16]}")
print(f"captured on: {capture['model']}")
print(f"replayed on: {MODEL}")

**Expect** about 900 calls and 25 journeys, captured on another deployment of
Nemotron 3.5 Super.

- Different models are fine: the load is the recorded one, call for call and
  reply length for reply length. Step 3's token check shows how close the
  prompts read.

## Step 3: Smoke Test and Checks

- Replays one short journey (J25), one conversation at a time.
- Checks the replay is faithful before any number counts:
  - **complete:** every conversation once, every call.
  - **in order:** each conversation's calls in recorded order.
  - **output lengths:** every reply as long as recorded (`ignore_eos`).
  - **prompt tokens:** share of calls the server counts within 2% of the capture.

In [ ]:
from stress_replay import RESULTS, expand, gates, replay, summarize, write_jsonl

smoke = expand(read_jsonl(TRACES / "journeys" / "J25.jsonl.xz"), copies=1, run="smoke")
smoke_dir = RESULTS / "smoke"
done = replay(write_jsonl(smoke_dir / "trace.jsonl", smoke), 1, 1, smoke_dir, BASE_URL, METRICS_URL, MODEL, KEY)

print("AIPerf exit code:", done.returncode, f"  (log: {smoke_dir / 'aiperf.out'})")
print(summarize(smoke_dir))
print(gates(smoke_dir, smoke, manifest))

**Expect:** exit code `0`, `errors: 0`, and all checks `True`, with
`prompt_tokens_within_2pct` near `1.0`.

- Exit code not `0`: read the log it names. `401`: the key is not the server's.
- `output_lengths: False`: replies ended early and every number describes
  less work. Likeliest on skill-selection calls, which force one tool
  (`tool_choice`) and so a fixed-length reply.
- `prompt_tokens_within_2pct` well below `1.0`: the server reads different
  prompts (another chat template). Re-capture with Notebook 4 against this
  server (`LLM_BASE_URL`).

## Step 4: Run the Matrix

- One run per concurrency and repeat. Every run plays each conversation once.
- **More shoppers than journeys:** above 25 at once, each journey is copied
  with fresh session and request ids. Copies share the system prompt and tools,
  like real shoppers, and part at the first shopper message.
- **Clean start without a restart:** every run renames all ids, so no run
  reuses the previous run's conversations. The system prompt and tools stay
  warm, as they would in production.
- **Concurrency above `MAX_NUM_SEQS`** (128 in `deploy.sh`) queues, and shows
  up as TTFT.
- Each run adds a row to `notebook/stress_results/summary.csv`. A run already
  there is skipped, so an interrupted matrix picks up where it stopped.
- **Live:** progress and the model's GPUs (busy, power, memory), redrawn every
  2 seconds; at the end, each GPU's average over the run. Each second is kept
  in the run's `gpu.csv`; averages go into the summary.
- **Cost:** two runs. For a no-load baseline add `1`: ~900 calls one after
  another, the slowest run of all. For a finer curve
  add `4` or `128`, or raise `REPEATS`; each is another run.

In [ ]:
from math import ceil

from stress_replay import append_summary, gpu_summary, read_summary

CONCURRENCY = [16, 64]
REPEATS = 1

finished = set()
if (RESULTS / "summary.csv").exists():
    finished = {(row["run"], row["model"]) for row in read_summary()}

for concurrency in CONCURRENCY:
    copies = max(1, ceil(2 * concurrency / len(journeys)))
    for repeat in range(REPEATS):
        run = f"cache-{CACHE}-b{BLOCK}-c{concurrency}-r{repeat}"
        if (run, MODEL) in finished:
            print(f"{run}: already in summary.csv, skipped")
            continue
        lines = expand(trace, copies=copies, run=run)
        sessions = len(journeys) * copies
        out = RESULTS / run
        done = replay(write_jsonl(out / "trace.jsonl", lines), sessions, concurrency, out,
                      BASE_URL, METRICS_URL, MODEL, KEY)
        if done.returncode:
            print(f"{run}: AIPerf failed, see {out / 'aiperf.out'}")
            continue
        row = {"run": run, "cache": CACHE, "block_size": BLOCK, "mamba_cache_mode": server["mamba_cache_mode"],
               "concurrency": concurrency, "repeat": repeat, "sessions": sessions, "model": MODEL,
               **summarize(out), **gpu_summary(out), **gates(out, lines, manifest)}
        append_summary(row)
        hit = f"{row['cache_hit_rate']:.0%}" if row["cache_hit_rate"] is not None else "n/a"
        gpu = (f"   GPU {row['gpu_busy_pct']:3.0f}% busy {row['gpu_power_w']:4.0f} W"
               if row["gpu_busy_pct"] is not None else "")
        print(f"{run:<26} hit {hit:>4}   TTFT p50 {row['ttft_p50_ms']:7.0f} ms  p95 {row['ttft_p95_ms']:7.0f} ms"
              f"   {row['requests_per_s']:5.1f} req/s{gpu}   complete {row['complete']}")

**Expect** one line per run. Hit rate should hold as concurrency rises, until
the cache runs out of room; TTFT p95 rises with load.

- `complete False`: the run is not the trace. Check `errors` in the summary
  and `docker logs nemotron35`.
- Crashes, not slowdowns, at high concurrency: a known MoE kernel issue.
  Redeploy with a lower `MAX_NUM_SEQS` and drop the levels above it.
- To stop early: interrupt the kernel. Finished runs are kept; rerunning the
  cell skips them.

## Step 5: Caching Off, and Block Size

- **Caching off**, what caching is worth: the same runs, recomputing every prompt.
  1. `PREFIX_CACHE=0 notebook/gpu/deploy.sh` (reloads from the local cache, minutes).
  2. Run Step 1: expect `prefix caching: off`.
  3. Run Step 4 again, with the same `CONCURRENCY`.
- **Block size** (optional): how finely prompts are matched.
  `BLOCK_SIZE=32 notebook/gpu/deploy.sh`, then Steps 1 and 4 as above.
- Redeploy with defaults when done: `notebook/gpu/deploy.sh`.

## Step 6: Results

- Median over repeats, per caching setting, block size, and concurrency.

In [ ]:
from collections import defaultdict
from statistics import median

from stress_replay import read_summary

rows = [row for row in read_summary() if row["model"] == MODEL]
groups = defaultdict(list)
for row in rows:
    groups[(f"{row['cache']}/b{row['block_size']}", int(row["concurrency"]))].append(row)

def mid(runs, field):
    values = [float(run[field]) for run in runs if run.get(field) not in ("", "None", None)]
    return median(values) if values else float("nan")

print(f"{'setting':<12}{'conc':>5}{'runs':>5}{'hit rate':>10}{'TTFT p50':>10}{'TTFT p95':>10}{'req/s':>8}{'out tok/s':>11}"
      f"{'GPU busy':>10}{'GPU W':>7}")
table = []
for (setting, concurrency), runs in sorted(groups.items()):
    entry = {"setting": setting, "concurrency": concurrency, **{f: mid(runs, f) for f in
             ("cache_hit_rate", "ttft_p50_ms", "ttft_p95_ms", "requests_per_s", "output_tokens_per_s",
              "gpu_busy_pct", "gpu_power_w")}}
    table.append(entry)
    show = lambda value, spec: "n/a" if value != value else format(value, spec)
    print(f"{setting:<12}{concurrency:>5}{len(runs):>5}{show(entry['cache_hit_rate'], '.0%'):>10}"
          f"{entry['ttft_p50_ms']:>10.0f}{entry['ttft_p95_ms']:>10.0f}{entry['requests_per_s']:>8.1f}"
          f"{entry['output_tokens_per_s']:>11.0f}{show(entry['gpu_busy_pct'] / 100, '.0%'):>10}"
          f"{show(entry['gpu_power_w'], '.0f'):>7}")

**Expect** caching on to show a high hit rate and a clearly lower TTFT than
caching off at the same concurrency.

- **TTFT is the proof, not the hit rate.** A high hit rate with TTFT no better
  than caching off means the Mamba state is being recomputed.
- Notebook 4's Step 6 estimated the ceiling from the text (about 70% within a
  conversation, 80% across). A measured rate well below it points at the chat
  template or cache eviction.
- **Is the GPU full?** `GPU busy` is the share of time any kernel ran, so it
  reads near 100% even for small batches. Power says more: well under the limit
  (700 W on H100) is headroom; near it is saturated. The other test is
  throughput: if a higher concurrency still raises `req/s`, the lower one was
  not using the GPU fully.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for setting in sorted({entry["setting"] for entry in table}):
    points = [entry for entry in table if entry["setting"] == setting]
    x = [entry["concurrency"] for entry in points]
    left.plot(x, [entry["cache_hit_rate"] for entry in points], marker="o", label=setting)
    right.plot(x, [entry["ttft_p95_ms"] for entry in points], marker="o", label=setting)
for axis, title in ((left, "Prefix-cache hit rate"), (right, "TTFT p95 (ms)")):
    axis.set_xscale("log", base=2)
    axis.set_xlabel("concurrent conversations")
    axis.set_title(title)
    axis.legend()
left.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()

## Next

- **Steady state:** one long run at the best level from Step 4, with more
  copies (raise `copies`), to see the hit rate hold once the cache is full.
- **End to end:** drive whole shopper journeys through the full assistant
  at two levels; the cache hit rate should agree with the figures here.
- **Results stay local:** `notebook/stress_results/` is ignored by git. Do not
  commit or publish measurements.